# Encoder–HistGB: encoder architecture and HistGB optimization

Reuse preprocessing and the starting HistGB parameters from `features_selection.ipynb`.
First search the encoder architecture, keeping HistGB parameters fixed; then tune HistGB
using the winning encoder settings passed directly through `winning_params`.
Inputs remain original imputed features, missingness indicators and elapsed time, with
bidirectional attention and no CLS token. HistGB receives engineered features plus embeddings.

1. **Encoder search:** 24 grid configurations, the first three of five fixed GroupKFold patient folds, seed 42.
2. **Confirmation:** best three configurations plus the incumbent, all five folds, seeds 42, 43 and 44.
3. **HistGB search:** cache the winner's embeddings on those five folds and three seeds, then run 40 trials without retraining the encoder.
4. **Optional final model:** train the winning encoder and tuned HistGB on all training patients with a fixed seed.

Both optimization stages select the epoch count using an internal 20% patient holdout
(maximum 150 epochs), followed by a fresh encoder fit on all fold-training patients.
Outer-validation patients train neither model. These reused folds provide tuning/comparison
scores, not an independent test estimate. There are no out-of-fold training embeddings.


## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Paths work from the project root or either notebook folder. N_TRIALS and TOP_K control encoder search and confirmation; HISTGB_N_TRIALS controls the final HistGB search.

In [1]:
import json
import os
import optuna
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)  # The reference setup reads data relative to the project.
    for cell_id in ["selection-imports", "selection-history", "selection-imputation",
                    "selection-features", "selection-pipeline"]:
        exec("".join(next(c for c in reference["cells"] if c.get("id") == cell_id)["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)  # Imputer helpers creation + one-hot → predictive imputation/flags → engineered features → median imputation/flags → no selection → tuned HistGB (unfitted).

# Start from the confirmed winner (trial 19) of the previous search.
ENCODER = dict(width=32, heads=8, layers=3, dropout=0.1347886862599206,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 150, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 0.001759366313629433, 4.3514588969246546e-05
SEEDS = [42, 43, 44]
N_TRIALS, TOP_K = 24, 3
HISTGB_N_TRIALS = 40
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

/Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. Build patients sequences (Encoder dataset)

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [2]:
def encoder_features(imputed):
    """Prepare per-visit inputs without patient IDs, adding elapsed time.

    Keep the original imputed features and missingness indicators. Compute
    elapsed years as age minus the patient's age at their first available visit.
    Patient IDs are used only for this grouping, not as encoder inputs.
    """
    
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    """Build one padded sequence per patient, with visits ordered by age.

    Features and targets must follow the same row order as raw. Return tensors
    of features [patients, visits, features], targets [patients, visits], and
    original row positions [patients, visits]. Padding positions are marked -1
    so attention and loss can ignore them; real positions restore embeddings
    to the original table order. Without targets, zero labels are unused
    placeholders for prediction.
    """
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Parameterized bidirectional encoder

Dense input projection → TransformerEncoder layers → one embedding per visit.
The grid varies joint width/head choices, encoder depth and regression-head depth. The head
contains two or three hidden layers; it supervises training but is not fed to HistGB.
Embedding width remains the Transformer width regardless of the regression head architecture.

In [3]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # 1. Initialize the encoder’s weights
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        # 2. Build the regression head
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [4]:
def train_network(training, seed, epochs, validation=None, params=None):
    torch.manual_seed(seed)
    params = {**ENCODER, "learning_rate": LEARNING_RATE,
              "weight_decay": WEIGHT_DECAY, **(params or {})}
    network = VisitEncoder(training.tensors[0].shape[-1],
                           **{key: params[key] for key in ENCODER}).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=params["learning_rate"],
                                 weight_decay=params["weight_decay"])
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed, params=None):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping, params=params)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs, params=params)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column.

In [5]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.projection.out_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Prepare the five disjoint patient folds once

GroupKFold uses five fixed folds with split seed 42. Preprocessing and HistGB baseline are fitted once per fold, then reused across
configurations. Search uses the first three splits; confirmation uses all five. Encoder early
stopping and refitting remain inside each fold and seed. No embeddings cross fold boundaries.

In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
prepared_folds = []
with threadpool_limits(limits=3):
    for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        prepared_folds.append(dict(
            raw_train=raw_train, raw_val=raw_val, yt=yt, yv=yv,
            train=train, validation=validation,
            imputed_train=prep[:2].transform(raw_train), imputed_val=prep[:2].transform(raw_val),
            baseline_mse=mean_squared_error(yv, original.predict(validation)),
        ))


def evaluate_config(params, folds, seeds, trial=None):
    scores = []
    with threadpool_limits(limits=3):
        for fold_number, fold in enumerate(folds, 1):
            for seed in seeds:
                fitted = fit_encoder(fold["raw_train"], fold["yt"], fold["imputed_train"], seed, params)
                z_train, _ = encoder_outputs(fitted, fold["raw_train"], fold["imputed_train"])
                z_val, prediction = encoder_outputs(fitted, fold["raw_val"], fold["imputed_val"])
                model = clone(baseline[-1]).fit(fold["train"].join(z_train), fold["yt"])
                mse = mean_squared_error(fold["yv"], model.predict(fold["validation"].join(z_val)))
                scores.append(dict(fold=fold_number, seed=seed, epochs=fitted["epochs"],
                                   HistGB=fold["baseline_mse"], Encoder_HistGB=mse,
                                   Transformer=mean_squared_error(fold["yv"], prediction),
                                   delta_MSE=mse - fold["baseline_mse"]))
                if trial is None:
                    print(f"Fold {fold_number}/{len(folds)}, seed {seed}: MSE {mse:.4f}")
            if trial is not None:
                trial.report(np.mean([s["Encoder_HistGB"] for s in scores]), step=fold_number)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return pd.DataFrame(scores)

## 7. Optuna search: three splits, seed 42

Search the 24 grid configurations, evaluating the confirmed incumbent first. The objective
is mean **Encoder–HistGB MSE**, not standalone encoder MSE. Unpromising trials can be pruned
after at least two completed folds. N_TRIALS can cap the number of grid configurations tested.

Search space: joint (width, heads) choices (16, 8), (24, 8), (32, 8), (32, 16);
3/4/5 encoder layers; 2/3 regression-head hidden layers. All pairs satisfy width % heads == 0.
Keep ff_multiplier=3, dropout=0.1347886862599206, mlp_units=16,
learning_rate=0.001759366313629433 and weight_decay=4.3514588969246546e-05 fixed.
These parameters showed no clear directional boundary signal in the previous search.
The epoch ceiling is now 150 because one of the strongest previous trials reached 100;
patience remains 10 and batch size remains 128.


In [7]:
WIDTH_HEADS = {"16_8": (16, 8), "24_8": (24, 8),
               "32_8": (32, 8), "32_16": (32, 16)}
SEARCH_SPACE = dict(width_heads=list(WIDTH_HEADS), layers=[3, 4, 5], mlp_layers=[2, 3])


def encoder_params(params):
    """Decode grid choices and include the fixed architecture/training settings."""
    width, heads = WIDTH_HEADS[params["width_heads"]]
    return {**ENCODER, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
            "width": width, "heads": heads, "layers": params["layers"],
            "mlp_layers": params["mlp_layers"]}


def suggest_params(trial):
    return encoder_params({name: trial.suggest_categorical(name, values)
                           for name, values in SEARCH_SPACE.items()})


def objective(trial):
    scores = evaluate_config(suggest_params(trial), prepared_folds[:3], [42], trial)
    trial.set_user_attr("epochs", scores["epochs"].tolist())
    return scores["Encoder_HistGB"].mean()


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.BruteForceSampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
initial_params = dict(width_heads=f"{ENCODER['width']}_{ENCODER['heads']}",
                      layers=ENCODER["layers"], mlp_layers=ENCODER["mlp_layers"])
study.enqueue_trial(initial_params)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
study.trials_dataframe().sort_values("value")

/var/folders/q8/d21qf_z93f37d2z9b04pqw3h0000gn/T/ipykernel_99838/3768232637.py:26: ExperimentalWarning: BruteForceSampler is experimental (supported from v3.1.0). The interface can change in the future.
  direction="minimize", sampler=optuna.samplers.BruteForceSampler(seed=42),
[I 2026-11-05 16:10:59,417] A new study created in memory with name: no-name-7f03d9ac-0bb1-41fa-bfad-0b77b6165280
Best trial: 0. Best value: 7.77764:   4%|▍         | 1/24 [04:33<1:44:55, 273.73s/it]

[I 2026-11-05 16:15:33,089] Trial 0 finished with value: 7.777641832450873 and parameters: {'width_heads': '32_8', 'layers': 3, 'mlp_layers': 2}. Best is trial 0 with value: 7.777641832450873.


Best trial: 1. Best value: 7.26887:   8%|▊         | 2/24 [13:29<2:36:55, 428.00s/it]

[I 2026-11-05 16:24:29,042] Trial 1 finished with value: 7.268873272775081 and parameters: {'width_heads': '24_8', 'layers': 5, 'mlp_layers': 3}. Best is trial 1 with value: 7.268873272775081.


Best trial: 1. Best value: 7.26887:   8%|▊         | 2/24 [18:01<2:36:55, 428.00s/it]

[I 2026-11-05 16:29:00,709] Trial 2 finished with value: 7.970055122427586 and parameters: {'width_heads': '32_8', 'layers': 3, 'mlp_layers': 3}. Best is trial 1 with value: 7.268873272775081.


... (39 outputs left out)

[I 2026-11-05 17:52:30,460] Trial 22 pruned. 


Best trial: 1. Best value: 7.26887: 100%|██████████| 24/24 [1:48:29<00:00, 271.22s/it]

[I 2026-11-05 17:59:28,691] Trial 23 finished with value: 7.335893930304413 and parameters: {'width_heads': '32_16', 'layers': 4, 'mlp_layers': 2}. Best is trial 1 with value: 7.268873272775081.


,number,value,datetime_start,datetime_complete,duration,params_layers,params_mlp_layers,params_width_heads,user_attrs_epochs,system_attrs_fixed_params,state
1,1,7.268873,2026-11-05 16:15:33.152728,2026-11-05 16:24:29.042480,0 days 00:08:55.889752,5,3,24_8,"[88, 79, 99]",NaN,COMPLETE
23,23,7.335894,2026-11-05 17:52:30.517254,2026-11-05 17:59:28.691288,0 days 00:06:58.174034,4,2,32_16,"[60, 80, 48]",NaN,COMPLETE
18,18,7.494293,2026-11-05 17:35:03.915985,2026-11-05 17:40:01.220780,0 days 00:04:57.304795,5,2,32_16,"[29, 91, 37]",NaN,COMPLETE
15,15,7.543213,2026-11-05 17:22:38.604109,2026-11-05 17:30:14.517656,0 days 00:07:35.913547,5,2,16_8,"[66, 99, 96]",NaN,COMPLETE
10,10,7.546820,2026-11-05 17:01:31.721407,2026-11-05 17:06:11.592544,0 days 00:04:39.871137,5,2,32_8,"[33, 44, 64]",NaN,COMPLETE
12,12,7.559450,2026-11-05 17:08:20.911744,2026-11-05 17:13:57.864279,0 days 00:05:36.952535,5,3,32_16,"[33, 72, 73]",NaN,COMPLETE
5,5,7.596160,2026-11-05 16:40:15.461471,2026-11-05 16:46:39.373148,0 days 00:06:23.911677,3,2,32_16,"[68, 49, 57]",NaN,COMPLETE
4,4,7.610805,2026-11-05 16:35:27.482328,2026-11-05 16:40:15.262531,0 days 00:04:47.780203,4,3,32_8,"[37, 36, 50]",NaN,COMPLETE
19,19,7.634154,2026-11-05 17:40:01.275926,2026-11-05 17:43:26.722652,0 days 00:03:25.446726,4,3,24_8,NaN,NaN,PRUNED
22,22,7.685568,2026-11-05 17:49:59.505433,2026-11-05 17:52:30.460786,0 days 00:02:30.955353,4,3,32_16,NaN,NaN,PRUNED


## 8. Confirm the best configurations: five splits, three seeds

Evaluate the best TOP_K completed configurations (default three), plus the incumbent if
not already selected, on all five splits with seeds 42, 43 and 44. Select the configuration with the lowest mean pipeline MSE
across these 15 runs, not the seed with the lowest score. Standard deviations describe variability.
These scores guide model selection; because search and confirmation reuse validation patients,
they are not an independent, unbiased estimate of the selected model's test performance.

In [8]:
candidates = sorted(
    (trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE),
    key=lambda trial: trial.value,
)[:TOP_K]
incumbent = study.trials[0]  # The confirmed configuration was enqueued first.
if all(candidate.number != incumbent.number for candidate in candidates):
    candidates.append(incumbent)
confirmation = []
for candidate in candidates:
    print(f"Confirming configuration from trial {candidate.number}")
    result = evaluate_config(encoder_params(candidate.params), prepared_folds, SEEDS)
    confirmation.append(result.assign(trial=candidate.number))
confirmation = pd.concat(confirmation, ignore_index=True)
confirmation_summary = confirmation.groupby("trial").agg(
    MSE=("Encoder_HistGB", "mean"), std=("Encoder_HistGB", lambda s: s.std(ddof=0)),
    HistGB=("HistGB", "mean"), delta_MSE=("delta_MSE", "mean"),
).sort_values("MSE")
winning_trial = int(confirmation_summary.index[0])
winning_params = next(encoder_params(trial.params) for trial in candidates
                      if trial.number == winning_trial)
scores = confirmation.loc[confirmation["trial"].eq(winning_trial)].copy()
print("Winning encoder parameters:", winning_params)
confirmation_summary

Confirming configuration from trial 1
Fold 1/5, seed 42: MSE 7.0249
Fold 1/5, seed 43: MSE 7.1964
Fold 1/5, seed 44: MSE 7.1232
Fold 2/5, seed 42: MSE 7.4740
Fold 2/5, seed 43: MSE 7.0677
Fold 2/5, seed 44: MSE 7.2801
Fold 3/5, seed 42: MSE 7.3077
Fold 3/5, seed 43: MSE 7.8670
Fold 3/5, seed 44: MSE 7.5449
Fold 4/5, seed 42: MSE 8.2942
Fold 4/5, seed 43: MSE 8.0310
Fold 4/5, seed 44: MSE 7.9357
Fold 5/5, seed 42: MSE 8.0193
Fold 5/5, seed 43: MSE 7.6040
Fold 5/5, seed 44: MSE 6.9270
Confirming configuration from trial 23
Fold 1/5, seed 42: MSE 7.2831
Fold 1/5, seed 43: MSE 7.5151
Fold 1/5, seed 44: MSE 7.2965
Fold 2/5, seed 42: MSE 7.0354
Fold 2/5, seed 43: MSE 7.7145
Fold 2/5, seed 44: MSE 7.8272
Fold 3/5, seed 42: MSE 7.6893
Fold 3/5, seed 43: MSE 7.7514
Fold 3/5, seed 44: MSE 7.4491
Fold 4/5, seed 42: MSE 8.0031
Fold 4/5, seed 43: MSE 7.6272
Fold 4/5, seed 44: MSE 7.8637
Fold 5/5, seed 42: MSE 7.1470
Fold 5/5, seed 43: MSE 7.2433
Fold 5/5, seed 44: MSE 7.3959
Confirming configuratio

,MSE,std,HistGB,delta_MSE
trial,,,,
1,7.513144,0.414592,11.73342,-4.220276
23,7.522773,0.277424,11.73342,-4.210647
0,7.700740,0.347612,11.73342,-4.032680
18,7.778840,0.466784,11.73342,-3.954581


In [9]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
HistGB,11.733420,0.307616,3.425408,0.000000
Transformer,8.219258,0.613819,2.866925,29.950025
Encoder_HistGB,7.513144,0.414592,2.741011,35.967998


## 9. Cache the winning encoder embeddings

Reuse the exact five patient folds and seeds 42, 43 and 44 from confirmation. Pass
`winning_params` directly to the same epoch-selection/refitting procedure and cache the
engineered features plus embeddings for each fold/seed. No architecture values are copied.


In [10]:
tuning_folds = []
with threadpool_limits(limits=3):
    for fold_number, fold in enumerate(prepared_folds, 1):
        for seed in SEEDS:
            fitted_encoder = fit_encoder(fold["raw_train"], fold["yt"],
                                         fold["imputed_train"], seed, winning_params)
            z_train, _ = encoder_outputs(fitted_encoder, fold["raw_train"], fold["imputed_train"])
            z_val, _ = encoder_outputs(fitted_encoder, fold["raw_val"], fold["imputed_val"])
            tuning_folds.append((fold["train"].join(z_train), fold["validation"].join(z_val),
                                 fold["yt"], fold["yv"]))
            print(f"Fold {fold_number}/5, seed {seed}: winning encoder refitted for "
                  f"{fitted_encoder['epochs']} selected epochs; embeddings cached.")


Fold 1/5, seed 42: winning encoder refitted for 88 selected epochs; embeddings cached.
Fold 1/5, seed 43: winning encoder refitted for 110 selected epochs; embeddings cached.
Fold 1/5, seed 44: winning encoder refitted for 72 selected epochs; embeddings cached.
Fold 2/5, seed 42: winning encoder refitted for 79 selected epochs; embeddings cached.
Fold 2/5, seed 43: winning encoder refitted for 123 selected epochs; embeddings cached.
Fold 2/5, seed 44: winning encoder refitted for 65 selected epochs; embeddings cached.
Fold 3/5, seed 42: winning encoder refitted for 99 selected epochs; embeddings cached.
Fold 3/5, seed 43: winning encoder refitted for 64 selected epochs; embeddings cached.
Fold 3/5, seed 44: winning encoder refitted for 56 selected epochs; embeddings cached.
Fold 4/5, seed 42: winning encoder refitted for 43 selected epochs; embeddings cached.
Fold 4/5, seed 43: winning encoder refitted for 137 selected epochs; embeddings cached.
Fold 4/5, seed 44: winning encoder refit

## 10. Optimize HistGB with the winning encoder

Run 40 Optuna trials on the 15 cached fold/seed datasets, starting with the previous
HistGB parameters. Only HistGB is retrained during this stage. These are tuning scores on
reused folds. The optional final cell uses the resulting `best_histgb`.


In [11]:
# Reuse the frozen embeddings cached above for all five splits and three seeds.
def histgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 800, step=100),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 7, 63, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 120, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 0, 30),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
    }
    scores = []
    for step, (train, validation, yt, yv) in enumerate(tuning_folds):
        fitted = clone(baseline[-1]).set_params(**params).fit(train, yt)
        scores.append(mean_squared_error(yv, fitted.predict(validation)))
        # Report after all seeds of a patient split; prune after at least two splits.
        if (step + 1) % len(SEEDS) == 0:
            trial.report(float(np.mean(scores)), step // len(SEEDS))
            if step < len(tuning_folds) - 1 and trial.should_prune():
                raise optuna.TrialPruned()
    trial.set_user_attr("fold_mse", scores)
    return float(np.mean(scores))


histgb_study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
)
parameter_names = ["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization", "max_features"]
histgb_study.enqueue_trial({name: baseline[-1].get_params()[name] for name in parameter_names})
with threadpool_limits(limits=3):
    histgb_study.optimize(histgb_objective, n_trials=HISTGB_N_TRIALS)

baseline_scores = histgb_study.trials[0].user_attrs["fold_mse"]
best_scores = histgb_study.best_trial.user_attrs["fold_mse"]
tuning_summary = pd.DataFrame([
    {"model": name, "MSE": np.mean(scores), "std": np.std(scores)}
    for name, scores in [("Encoder_HistGB_current", baseline_scores), ("Encoder_HistGB_tuned", best_scores)]
]).set_index("model")
tuning_summary["RMSE"] = np.sqrt(tuning_summary["MSE"])
tuning_summary["gain_MSE_pct"] = 100 * (1 - tuning_summary["MSE"] / np.mean(baseline_scores))
best_histgb = clone(baseline[-1]).set_params(**histgb_study.best_params)
print("Best HistGB parameters:", histgb_study.best_params)
tuning_summary


[I 2026-11-05 20:31:11,802] A new study created in memory with name: no-name-d8f3dde3-7c37-494b-9ea3-2d604b1521e7
[I 2026-11-05 20:33:04,739] Trial 0 finished with value: 7.513143857317598 and parameters: {'learning_rate': 0.042592090635273853, 'max_iter': 800, 'max_leaf_nodes': 62, 'min_samples_leaf': 25, 'l2_regularization': 1.8115270802885095, 'max_features': 0.6078232265551354}. Best is trial 0 with value: 7.513143857317598.
[I 2026-11-05 20:34:44,755] Trial 1 finished with value: 7.573363935168277 and parameters: {'learning_rate': 0.047377279007281566, 'max_iter': 800, 'max_leaf_nodes': 34, 'min_samples_leaf': 43, 'l2_regularization': 4.680559213273096, 'max_features': 0.662397808134481}. Best is trial 0 with value: 7.513143857317598.
[I 2026-11-05 20:36:07,751] Trial 2 finished with value: 7.638466404388142 and parameters: {'learning_rate': 0.022861967752626437, 'max_iter': 700, 'max_leaf_nodes': 26, 'min_samples_leaf': 57, 'l2_regularization': 0.6175348288740734, 'max_features':

Best HistGB parameters: {'learning_rate': 0.030423302996109857, 'max_iter': 800, 'max_leaf_nodes': 62, 'min_samples_leaf': 15, 'l2_regularization': 2.181108146133318, 'max_features': 0.6127351469608062}


,MSE,std,RMSE,gain_MSE_pct
model,,,,
Encoder_HistGB_current,7.513144,0.414592,2.741011,0.000000
Encoder_HistGB_tuned,7.496464,0.411907,2.737967,0.222007


## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Replace the Transformer with another encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed HistGB the encoder's final prediction instead of, or alongside, its embeddings.